In [0]:
from pyspark.sql.functions import col, sum, count, avg, round

spark.conf.set(
    "SUA_CHAVE_AQUI"
)

# 1. Ler o Silver
df_silver = spark.read.format("delta") \
    .load("SUA_CHAVE_AQUI/vendas/silver/vendas")

# 2. Criar banco de dados no metastore
spark.sql("CREATE DATABASE IF NOT EXISTS adb_ecommerce")

# 3. Criar e salvar tabelas Gold direto no metastore

# Por categoria
df_gold_categoria = df_silver.groupBy("categoria") \
    .agg(
        count("order_id").alias("qtd_pedidos"),
        round(sum("receita_liquida"), 2).alias("total_receita"),
        round(avg("receita_liquida"), 2).alias("ticket_medio")
    ).orderBy("total_receita", ascending=False)

df_gold_categoria.write.format("delta").mode("overwrite") \
    .saveAsTable("adb_ecommerce.gold_categoria")

# Por canal
df_gold_canal = df_silver.groupBy("canal_venda") \
    .agg(
        count("order_id").alias("qtd_pedidos"),
        round(sum("receita_liquida"), 2).alias("total_receita")
    ).orderBy("total_receita", ascending=False)

df_gold_canal.write.format("delta").mode("overwrite") \
    .saveAsTable("adb_ecommerce.gold_canal")

# Por estado
df_gold_estado = df_silver.groupBy("estado") \
    .agg(
        count("order_id").alias("qtd_pedidos"),
        round(sum("receita_liquida"), 2).alias("total_receita")
    ).orderBy("total_receita", ascending=False)

df_gold_estado.write.format("delta").mode("overwrite") \
    .saveAsTable("adb_ecommerce.gold_estado")

print("✅ Tabelas registradas no metastore com sucesso!")

✅ Tabelas registradas no metastore com sucesso!


In [0]:
from pyspark.sql.functions import count

spark.conf.set(
    "SUA_CHAVE_AQUI"
)

# Ler o Silver primeiro
df_silver = spark.read.format("delta") \
    .load("SUA_CHAVE_AQUI/vendas/silver/vendas")

# Criar tabela de status
df_gold_taxa = df_silver.groupBy("status_pedido") \
    .agg(count("order_id").alias("qtd")) \
    .orderBy("qtd", ascending=False)

df_gold_taxa.write.format("delta").mode("overwrite") \
    .saveAsTable("adb_ecommerce.gold_status")

print("✅ Tabela gold_status criada com sucesso!")
display(df_gold_taxa)

✅ Tabela gold_status criada com sucesso!


status_pedido,qtd
Entregue,229
Em trânsito,97
Devolvido,86


In [0]:
from pyspark.sql.functions import col, sum, count, lit

spark.conf.set(
    "SUA_CHAVE_AQUI"
)

df_silver = spark.read.format("delta") \
    .load("SUA_CHAVE_AQUI/vendas/silver/vendas")

df_gold_estado = df_silver.groupBy("estado") \
    .agg(
        count("order_id").alias("qtd_pedidos"),
        sum("receita_liquida").alias("total_receita")
    ) \
    .withColumn("pais", lit("Brazil"))

df_gold_estado.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("adb_ecommerce.gold_estado")

print("✅ Tabela atualizada com sucesso!")
display(df_gold_estado)

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File <command-7551703538959711>, line 3
      1 from pyspark.sql.functions import col, sum, count, lit
----> 3 spark.conf.set(
      4     "SUA_CHAVE_AQUI"
      5 )
      7 df_silver = spark.read.format("delta") \
      8     .load("SUA_CHAVE_AQUI/vendas/silver/vendas")
     10 df_gold_estado = df_silver.groupBy("estado") \
     11     .agg(
     12         count("order_id").alias("qtd_pedidos"),
     13         sum("receita_liquida").alias("total_receita")
     14     ) \
     15     .withColumn("pais", lit("Brazil"))

TypeError: RuntimeConf.set() missing 1 required positional argument: 'value'